## **Tugas Praktikum Big Data 11 (Florania Ayodya Firdyaziz)**

### **Mempersiapkan SparkSession dan Membaca Dataset**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas11-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.show(5)

print("Total baris: ", df.count())

26/11/15 16:29:03 WARN Utils: Your hostname, floo resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/15 16:29:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 16:29:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



Total baris:  2500


### **A. Feature Engineering**

In [4]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

model_menu = indexer_menu.fit(df)
df_indexed = model_menu.transform(df)
model_cabang = indexer_cabang.fit(df_indexed)
df_indexed = model_cabang.transform(df_indexed)
model_waktu = indexer_waktu.fit(df_indexed)
df_indexed = model_waktu.transform(df_indexed)

assembler = VectorAssembler(
    inputCols=[
        "qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"
    ],
    outputCol="features"
)

print("Pengubahan menu, cabang, dan waktu menjadi index numerik")
df_indexed.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_pesan_idx").show(5)

df_final = assembler.transform(df_indexed)
df_final.select("features", "total_penjualan").show(5, truncate=False)

Pengubahan menu, cabang, dan waktu menjadi index numerik
+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
+--------------+-------------+----------+----------+-----------+---------------+
only showing top 5 rows

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|

### **B. Train/Test Split & Pelatihan Model**

In [6]:
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print("Jumlah data training: ", train.count())
print("Jumlah data testing: ", test.count())

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)
print("Model berhasil dilatih")
print("Koefisien: ", model.coefficients)
print("Intercept: ", model.intercept)

Jumlah data training:  1839


Jumlah data testing:  661


[Stage 43:>                                                         (0 + 1) / 1]

Model berhasil dilatih
Koefisien:  [22046.663271025755,3.0229208681606696,-123.21425598657828,279.2845246167609,339.63879532317213]
Intercept:  -66903.1776648588


### **C. Evaluasi Model**

In [8]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)
predictions.select("total_penjualan", "prediction").show(10)

evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)
rmse = evaluator_rmse.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R2 : {r2:,.4f}")

+------------------+------------------+
|   total_penjualan|        prediction|
+------------------+------------------+
|19721.170446051627|31056.146105506865|
|  21236.0956379149|31056.146105506865|
|  85834.8564401191| 88127.37331410211|
|102174.09100317833| 88127.37331410211|
|113044.63431555482|112310.74025938749|
|115810.29600214415|110174.03658512786|
|140386.40273615732|119242.79918960987|
| 73842.64420790318|  67877.7749220128|
|32345.642983082646| 44580.62517062851|
| 45510.16712269087| 66627.28844165425|
+------------------+------------------+
only showing top 10 rows



RMSE : Rp 10,901
R2 : 0.9107


**Interprestasi hasil:**

Nilai RMSE menunjukkan rata-rata kesalahan prediksi model dalam satuan Rupiah, sedangkan R2 menunjukkan seberapa baik model menjelaskan variasi total penjualan. Model dapat dikatakan memiliki performa yang lebih baik apabila nilai RMSE relatif kecil dan R2 mendekati 1.

### **D. Eksperimen Perbandingan Fitur**

In [12]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

assembler_sederhana = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_sederhana")
df_sederhana = assembler_sederhana.transform(df_indexed)

train_sederhana, test_sederhana = df_sederhana.randomSplit([0.75, 0.25], seed=101)
lr_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model_sederhana = lr_sederhana.fit(train_sederhana)
predictions_sederhana = model_sederhana.transform(test_sederhana)

evaluator_sederhana = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="r2"
)
r2_sederhana = evaluator_sederhana.evaluate(predictions_sederhana)

print(f"R2 Model Utama : {r2:,.4f}")
print(f"R2 Model Sederhana: {r2_sederhana:,.4f}")

if r2 > r2_sederhana:
    print("Model sederhana memiliki akurasi lebih baik")
elif r2 < r2_sederhana:
    print("Model sederhana memiliki akurasi lebih buruk")
else:
    print("Model sederhana memiliki akurasi yang sama")

R2 Model Utama : 0.9107
R2 Model Sederhana: 0.9108
Model sederhana memiliki akurasi lebih buruk


**Kesimpulan:**

Perbandingan kedua model menunjukkan pengaruh penambahan informasi menu, cabang, dan waktu pemesanan terhadap hasil prediksi. Apabila model utama lebih tinggi, berarti penambahan fitur tersebut membantu meningkatkan kemampuan model dalam memprediksi total penjualan.

### **E. Prediksi Skenario Baru**

In [13]:
data_baru = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 18000),
    ("Americano", "Yogyakarta", "Siang", 3, 20000),
    ("Cappuccino", "Semarang", "Malam", 4, 25000),
]

kolom = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_baru = spark.createDataFrame(data_baru, kolom)
df_baru_indexed = model_menu.transform(df_baru)
df_baru_indexed = model_cabang.transform(df_baru_indexed)
df_baru_indexed = model_waktu.transform(df_baru_indexed)

df_baru_final = assembler.transform(df_baru_indexed)
prediksi_baru = model.transform(df_baru_final)

prediksi_baru.select(
    "nama_menu",
    "cabang",
    "waktu_pesan",
    "qty",
    "harga_satuan",
    "prediction"
).show(3, truncate=False)

[Stage 50:>                                                         (0 + 1) / 1]

+----------+----------+-----------+---+------------+------------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+----------+----------+-----------+---+------------+------------------+
|Kopi Susu |Magelang  |Pagi       |2  |18000       |31479.510248098188|
|Americano |Yogyakarta|Siang      |3  |20000       |60653.79162669496 |
|Cappuccino|Semarang  |Malam      |4  |25000       |97508.27645584448 |
+----------+----------+-----------+---+------------+------------------+



### **Eksplorasi**

In [14]:
from pyspark.sql.functions import abs, col

analisis_error = predictions.withColumn(
    "selisih_prediksi",
    abs(col("total_penjualan") - col("prediction"))
)

analisis_error.select(
    "total_penjualan",
    "prediction",
    "selisih_prediksi"
).show(10, truncate=False)

analisis_error.selectExpr(
    "AVG(selisih_prediksi) AS rata_rata_error",
    "MAX(selisih_prediksi) AS error_terbesar",
    "MIN(selisih_prediksi) AS error_terkecil"
).show()

+------------------+------------------+------------------+
|total_penjualan   |prediction        |selisih_prediksi  |
+------------------+------------------+------------------+
|19721.170446051627|31056.146105506865|11334.975659455238|
|21236.0956379149  |31056.146105506865|9820.050467591966 |
|85834.8564401191  |88127.37331410211 |2292.5168739830115|
|102174.09100317833|88127.37331410211 |14046.71768907622 |
|113044.63431555482|112310.74025938749|733.8940561673371 |
|115810.29600214415|110174.03658512786|5636.259417016292 |
|140386.40273615732|119242.79918960987|21143.603546547456|
|73842.64420790318 |67877.7749220128  |5964.869285890381 |
|32345.642983082646|44580.62517062851 |12234.982187545866|
|45510.16712269087 |66627.28844165425 |21117.121318963378|
+------------------+------------------+------------------+
only showing top 10 rows



[Stage 52:>                                                         (0 + 1) / 1]

+-----------------+-----------------+------------------+
|  rata_rata_error|   error_terbesar|    error_terkecil|
+-----------------+-----------------+------------------+
|8733.627431923069|38788.48110067788|1.1053594550903654|
+-----------------+-----------------+------------------+



In [15]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr_regularisasi = LinearRegression(
    featuresCol="features",
    labelCol="total_penjualan",
    regParam=0.1,
    elasticNetParam=0.5
)

model_regularisasi = lr_regularisasi.fit(train)

prediksi_regularisasi = model_regularisasi.transform(test)

evaluator = RegressionEvaluator(
    labelCol="total_penjualan",
    predictionCol="prediction",
    metricName="rmse"
)

rmse_regularisasi = evaluator.evaluate(
    prediksi_regularisasi
)

print("RMSE Model Awal:", rmse)
print("RMSE Model Regularisasi:", rmse_regularisasi)

RMSE Model Awal: 10900.803581280326
RMSE Model Regularisasi: 10900.798290638235


In [16]:
spark.stop()
print("Spark berhasil dihentikan")

Spark berhasil dihentikan
